# Evaluasi Retrieval dan RAGAS

Dataset aktif: 64 pertanyaan dari delapan emiten pada `data/evaluation/user_benchmark_64`.
Kolom data acuan menjadi referensi. Jawaban pengujian ulang dibentuk dari enam konteks retrieval; jawaban awal disimpan sebagai pembanding.
Jalankan Run All di VS Code. RAGAS memerlukan koneksi API. `ULANG_RAGAS = True` membuat pengujian baru.


In [1]:
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from IPython.display import Markdown, display

ULANG_RAGAS = False
RAGAS_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
PYTHON_EVALUASI = ""
PYTHON_EMBEDDING = ""

PROJECT = next(
    candidate
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    for candidate in (parent, parent / "Program")
    if (candidate / "config.py").is_file()
    and (candidate / "evaluation/notebook_evaluation.py").is_file()
)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from evaluation.notebook_evaluation import (
    markdown_table, ragas_context_table, ragas_tables, read_json, read_jsonl,
    retrieval_question_tables, retrieval_tables, write_csv,
)

DATASET = PROJECT / "data/evaluation/user_benchmark_64"
EMBEDDING_CACHE = PROJECT / "evaluation/cache/user_bge_m3_64"
HASIL = PROJECT / "evaluation/results/evaluasi_retrieval_ragas_64"
BUNDLED_PYTHON = Path.home() / ".cache/codex-runtimes/codex-primary-runtime/dependencies/python/python.exe"
candidates = [Path(PYTHON_EVALUASI)] if PYTHON_EVALUASI else [
    PROJECT / ".venv/Scripts/python.exe", PROJECT / ".venv/bin/python",
    Path(sys.executable), BUNDLED_PYTHON,
]
for candidate in dict.fromkeys(candidates):
    if not candidate.is_file():
        continue
    environment = os.environ.copy()
    environment.pop("PYTHONPATH", None)
    environment["PYTHONUTF8"] = "1"
    environment["RAGAS_DO_NOT_TRACK"] = "true"
    environment["ANONYMIZED_TELEMETRY"] = "false"
    if candidate.resolve() == BUNDLED_PYTHON.resolve():
        environment["PYTHONPATH"] = os.pathsep.join(
            str(PROJECT / name) for name in (".evaluation-deps", ".python-deps")
            if (PROJECT / name).is_dir()
        )
    probe = subprocess.run(
        [str(candidate), "-c", "import sys, importlib.util; print(sys.version_info[:2] == (3, 12) and all(importlib.util.find_spec(m) for m in ('numpy', 'chromadb', 'rank_bm25', 'ragas', 'instructor', 'openai')))"] ,
        cwd=PROJECT, env=environment, capture_output=True, text=True,
        encoding="utf-8", errors="replace", timeout=15,
    )
    if probe.returncode == 0 and probe.stdout.strip() == "True":
        PYTHON, ENVIRONMENT = candidate, environment
        break
else:
    raise RuntimeError("Gunakan Python 3.12 dengan requirements.txt dan requirements-evaluation.txt. Isi PYTHON_EVALUASI jika interpreter berada di lokasi lain.")

questions = read_jsonl(DATASET / "questions.jsonl")
references = read_jsonl(DATASET / "ground_truth.jsonl")
registry = read_json(PROJECT / "data/registry/index_registry.json")
HASIL.mkdir(parents=True, exist_ok=True)

def jalankan(*arguments, allow_incomplete=False):
    result = subprocess.run(
        [str(PYTHON), "-X", "utf8", "-u", *map(str, arguments)],
        cwd=PROJECT, env=ENVIRONMENT, capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if result.returncode and not (allow_incomplete and result.returncode == 2):
        message = "\n".join((result.stdout + "\n" + result.stderr).splitlines()[-20:])
        raise RuntimeError(message)
    return result

def siapkan_embedding():
    if (EMBEDDING_CACHE / "manifest.json").is_file():
        return
    interpreters = [Path(PYTHON_EMBEDDING)] if PYTHON_EMBEDDING else [
        Path.home() / "AppData/Local/Programs/Python/Python311/python.exe",
        PYTHON, Path(sys.executable),
    ]
    for candidate in dict.fromkeys(interpreters):
        if not candidate.is_file():
            continue
        embedding_environment = os.environ.copy()
        embedding_environment.pop("PYTHONPATH", None)
        embedding_environment["PYTHONUTF8"] = "1"
        probe = subprocess.run(
            [str(candidate), "-c", "import importlib.util; print(all(importlib.util.find_spec(m) for m in ('torch', 'sentence_transformers', 'numpy', 'dotenv')))"],
            cwd=PROJECT, env=embedding_environment, capture_output=True, text=True,
            encoding="utf-8", errors="replace", timeout=15,
        )
        if probe.returncode == 0 and probe.stdout.strip() == "True":
            result = subprocess.run(
                [str(candidate), "-X", "utf8", "-u", "-m", "scripts.offline_query_embeddings",
                 "--questions", str(DATASET / "questions.jsonl"), "--output-dir", str(EMBEDDING_CACHE)],
                cwd=PROJECT, env=embedding_environment, capture_output=True, text=True,
                encoding="utf-8", errors="replace",
            )
            if result.returncode:
                raise RuntimeError("\n".join((result.stdout + result.stderr).splitlines()[-20:]))
            return
    raise RuntimeError("Isi PYTHON_EMBEDDING dengan interpreter yang memiliki torch dan sentence-transformers. Model BGE-M3 harus tersedia di cache lokal.")


## Hasil Pengujian Retrieval

In [2]:
siapkan_embedding()
jalankan(
    "-m", "evaluation.notebook_evaluation", "--output", HASIL / "retrieval.json",
    "--questions", DATASET / "questions.jsonl", "--ground-truth", DATASET / "ground_truth.jsonl",
    "--embedding-cache", EMBEDDING_CACHE,
)
retrieval = read_json(HASIL / "retrieval.json")
distribusi, keseluruhan, kategori = retrieval_tables(questions, references, registry, retrieval)


### Tabel L12.1 Distribusi Dokumen dan Pertanyaan Pengujian Retrieval

In [3]:
display(Markdown(markdown_table(distribusi)))


| Emiten | Jumlah chunk | Pertanyaan dengan jawaban | Pertanyaan tanpa jawaban |
| --- | --- | --- | --- |
| BACH | 2.026 | 8 | 0 |
| EMMI | 1.730 | 8 | 0 |
| JECX | 2.616 | 8 | 0 |
| JELI | 1.813 | 8 | 0 |
| PRDL | 1.908 | 8 | 0 |
| RANS | 3.007 | 8 | 0 |
| SWAP | 2.599 | 8 | 0 |
| WBSA | 2.173 | 8 | 0 |
| Total | 17.872 | 64 | 0 |

### Tabel L12.2 Hasil Pengujian Retrieval Keseluruhan

In [4]:
display(Markdown(markdown_table(keseluruhan)))


| Metode | K | Precision@K | Recall@K | MRR@K |
| --- | --- | --- | --- | --- |
| Sparse Retrieval | 1 | 0,3906 | 0,0375 | 0,3906 |
| Sparse Retrieval | 3 | 0,3125 | 0,0864 | 0,4766 |
| Sparse Retrieval | 6 | 0,2969 | 0,1656 | 0,5169 |
| Dense Retrieval | 1 | 0,5625 | 0,0546 | 0,5625 |
| Dense Retrieval | 3 | 0,4896 | 0,1408 | 0,6484 |
| Dense Retrieval | 6 | 0,4115 | 0,2284 | 0,6745 |
| Hybrid Retrieval with RRF | 1 | 0,5000 | 0,0518 | 0,5000 |
| Hybrid Retrieval with RRF | 3 | 0,4844 | 0,1431 | 0,6302 |
| Hybrid Retrieval with RRF | 6 | 0,3958 | 0,2225 | 0,6586 |

### Tabel L12.3 Hasil Retrieval Berdasarkan Kategori Pertanyaan pada K = 6

In [5]:
display(Markdown(markdown_table(kategori)))


| Kategori | Metode | Precision@6 | Recall@6 | MRR@6 |
| --- | --- | --- | --- | --- |
| Fakta eksplisit | Sparse | 0,3636 | 0,1841 | 0,6106 |
| Fakta eksplisit | Dense | 0,4773 | 0,2602 | 0,7765 |
| Fakta eksplisit | Hybrid | 0,4848 | 0,2643 | 0,7705 |
| Parafrasa | Sparse | 0,4035 | 0,2244 | 0,6377 |
| Parafrasa | Dense | 0,3947 | 0,1471 | 0,5965 |
| Parafrasa | Hybrid | 0,4298 | 0,2028 | 0,7018 |
| Tabel numerik | Sparse | 0,1538 | 0,1300 | 0,3038 |
| Tabel numerik | Dense | 0,4359 | 0,3812 | 0,7115 |
| Tabel numerik | Hybrid | 0,3205 | 0,2731 | 0,5538 |
| Lintas bagian | Sparse | 0,1333 | 0,0594 | 0,3583 |
| Lintas bagian | Dense | 0,2667 | 0,1146 | 0,5500 |
| Lintas bagian | Hybrid | 0,2333 | 0,1021 | 0,4667 |

### Hasil Pengujian Retrieval per Pertanyaan

In [6]:
teks_chunk = read_json(HASIL / "retrieval_chunks.json")
rincian_retrieval, peringkat_retrieval = retrieval_question_tables(questions, references, retrieval, teks_chunk)
kolom_retrieval = ["No", "Emiten", "Pertanyaan", "Metode", "K", "Precision@K", "Recall@K", "MRR@K"]
nama_metode = {"Sparse Retrieval": "Sparse", "Dense Retrieval": "Dense", "Hybrid Retrieval with RRF": "Hybrid"}
tabel_retrieval = [
    {kolom: nama_metode.get(row[kolom], row[kolom]) if kolom == "Metode" else row[kolom]
     for kolom in kolom_retrieval}
    for row in rincian_retrieval
]
display(Markdown(markdown_table(tabel_retrieval)))


| No | Emiten | Pertanyaan | Metode | K | Precision@K | Recall@K | MRR@K |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Sparse | 6 | 0,1667 | 0,0714 | 0,2000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Dense | 3 | 0,3333 | 0,0714 | 0,5000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Dense | 6 | 0,3333 | 0,1429 | 0,5000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Hybrid | 3 | 0,3333 | 0,0714 | 0,5000 |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Hybrid | 6 | 0,3333 | 0,1429 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Sparse | 3 | 0,3333 | 0,0714 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Sparse | 6 | 0,1667 | 0,0714 | 0,5000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Hybrid | 6 | 0,3333 | 0,1429 | 0,2500 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Sparse | 6 | 0,3333 | 0,5000 | 0,2000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Dense | 3 | 1,0000 | 0,7500 | 1,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Dense | 6 | 0,5000 | 0,7500 | 1,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Hybrid | 3 | 1,0000 | 0,7500 | 1,0000 |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Hybrid | 6 | 0,5000 | 0,7500 | 1,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Sparse | 1 | 1,0000 | 0,2500 | 1,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Sparse | 3 | 0,6667 | 0,5000 | 1,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Sparse | 6 | 0,3333 | 0,5000 | 1,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Hybrid | 3 | 0,3333 | 0,2500 | 0,3333 |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Hybrid | 6 | 0,3333 | 0,5000 | 0,3333 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Sparse | 3 | 0,6667 | 0,2500 | 0,5000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Sparse | 6 | 0,3333 | 0,2500 | 0,5000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Dense | 3 | 0,3333 | 0,1250 | 0,5000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Dense | 6 | 0,3333 | 0,2500 | 0,5000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Hybrid | 3 | 0,3333 | 0,1250 | 0,3333 |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Hybrid | 6 | 0,3333 | 0,2500 | 0,3333 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 1,0000 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,3333 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,1667 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,3333 | 0,1250 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,5000 | 0,3750 | 1,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,6667 | 0,2500 | 0,5000 |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,3333 | 0,2500 | 0,5000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Sparse | 1 | 1,0000 | 0,0769 | 1,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Sparse | 3 | 0,6667 | 0,1538 | 1,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Sparse | 6 | 0,5000 | 0,2308 | 1,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Dense | 6 | 0,1667 | 0,0769 | 0,2500 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Hybrid | 1 | 1,0000 | 0,0769 | 1,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Hybrid | 3 | 0,6667 | 0,1538 | 1,0000 |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Hybrid | 6 | 0,3333 | 0,1538 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Sparse | 1 | 1,0000 | 0,0455 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Sparse | 3 | 1,0000 | 0,1364 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Sparse | 6 | 1,0000 | 0,2727 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Dense | 1 | 1,0000 | 0,0455 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Dense | 3 | 1,0000 | 0,1364 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Dense | 6 | 0,8333 | 0,2273 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Hybrid | 1 | 1,0000 | 0,0455 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Hybrid | 3 | 1,0000 | 0,1364 | 1,0000 |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Hybrid | 6 | 1,0000 | 0,2727 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Sparse | 3 | 0,6667 | 0,1333 | 0,5000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Sparse | 6 | 0,6667 | 0,2667 | 0,5000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Dense | 3 | 1,0000 | 0,2000 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Dense | 6 | 0,6667 | 0,2667 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Hybrid | 3 | 1,0000 | 0,2000 | 1,0000 |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Hybrid | 6 | 1,0000 | 0,4000 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Sparse | 3 | 0,6667 | 0,1333 | 0,5000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Sparse | 6 | 0,5000 | 0,2000 | 0,5000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Dense | 3 | 0,6667 | 0,1333 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Dense | 6 | 0,5000 | 0,2000 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Hybrid | 3 | 0,6667 | 0,1333 | 1,0000 |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Hybrid | 6 | 0,5000 | 0,2000 | 1,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Dense | 1 | 1,0000 | 0,0303 | 1,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Dense | 3 | 0,6667 | 0,0606 | 1,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Dense | 6 | 0,6667 | 0,1212 | 1,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Hybrid | 3 | 0,6667 | 0,0606 | 0,5000 |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Hybrid | 6 | 0,3333 | 0,0606 | 0,5000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Sparse | 1 | 1,0000 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Sparse | 3 | 0,3333 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Sparse | 6 | 0,3333 | 0,2857 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Dense | 1 | 1,0000 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Dense | 3 | 0,3333 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Dense | 6 | 0,3333 | 0,2857 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Hybrid | 1 | 1,0000 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Hybrid | 3 | 0,3333 | 0,1429 | 1,0000 |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Hybrid | 6 | 0,3333 | 0,2857 | 1,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,3333 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,1667 | 0,0909 | 1,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,1667 | 0,0909 | 0,2500 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,0909 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 1,0000 | 0,2727 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,6667 | 0,3636 | 1,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,3333 | 0,0909 | 0,3333 |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,5000 | 0,2727 | 0,3333 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 6 | 0,3333 | 0,0500 | 0,2500 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 6 | 0,1667 | 0,0250 | 0,2500 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 3 | 0,3333 | 0,0556 | 0,3333 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 6 | 0,5000 | 0,1667 | 0,3333 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 1 | 1,0000 | 0,0556 | 1,0000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 3 | 0,6667 | 0,1111 | 1,0000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 6 | 0,3333 | 0,1111 | 1,0000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 3 | 0,3333 | 0,0556 | 0,5000 |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 6 | 0,3333 | 0,1111 | 0,5000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Sparse | 1 | 1,0000 | 0,0714 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Sparse | 3 | 0,3333 | 0,0714 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Sparse | 6 | 0,3333 | 0,1429 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Dense | 1 | 1,0000 | 0,0714 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Dense | 3 | 0,3333 | 0,0714 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Dense | 6 | 0,3333 | 0,1429 | 1,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Hybrid | 3 | 0,3333 | 0,0714 | 0,5000 |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Hybrid | 6 | 0,3333 | 0,1429 | 0,5000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Sparse | 1 | 1,0000 | 0,0714 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Sparse | 3 | 0,3333 | 0,0714 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Sparse | 6 | 0,5000 | 0,2143 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Dense | 1 | 1,0000 | 0,0714 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Dense | 3 | 1,0000 | 0,2143 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Dense | 6 | 0,6667 | 0,2857 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Hybrid | 1 | 1,0000 | 0,0714 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Hybrid | 3 | 0,3333 | 0,0714 | 1,0000 |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Hybrid | 6 | 0,5000 | 0,2143 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Sparse | 6 | 0,1667 | 0,1111 | 0,2500 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Dense | 3 | 1,0000 | 0,3333 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Dense | 6 | 0,5000 | 0,3333 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Hybrid | 3 | 1,0000 | 0,3333 | 1,0000 |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Hybrid | 6 | 0,5000 | 0,3333 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Sparse | 1 | 1,0000 | 0,2500 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Sparse | 3 | 0,3333 | 0,2500 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Sparse | 6 | 0,1667 | 0,2500 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Dense | 6 | 0,1667 | 0,2500 | 0,2500 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Hybrid | 3 | 0,3333 | 0,2500 | 1,0000 |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Hybrid | 6 | 0,1667 | 0,2500 | 1,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,3333 | 0,2000 | 0,3333 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,3333 | 0,4000 | 0,3333 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,1667 | 0,2000 | 0,2000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,1250 | 1,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,6667 | 0,2500 | 1,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,5000 | 0,3750 | 1,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,3333 | 0,1250 | 0,5000 |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,1667 | 0,1250 | 0,5000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Sparse | 6 | 0,1667 | 0,0435 | 0,1667 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Dense | 1 | 1,0000 | 0,0435 | 1,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Dense | 3 | 0,3333 | 0,0435 | 1,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Dense | 6 | 0,5000 | 0,1304 | 1,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Hybrid | 3 | 0,3333 | 0,0435 | 0,5000 |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Hybrid | 6 | 0,3333 | 0,0870 | 0,5000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 1 | 1,0000 | 0,0833 | 1,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 3 | 0,3333 | 0,0833 | 1,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 6 | 0,1667 | 0,0833 | 1,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 6 | 0,1667 | 0,0833 | 0,2500 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 3 | 0,3333 | 0,0833 | 0,5000 |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 6 | 0,1667 | 0,0833 | 0,5000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Sparse | 1 | 1,0000 | 0,0714 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Sparse | 3 | 0,6667 | 0,1429 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Sparse | 6 | 0,8333 | 0,3571 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Dense | 1 | 1,0000 | 0,0714 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Dense | 3 | 1,0000 | 0,2143 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Dense | 6 | 0,5000 | 0,2143 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Hybrid | 1 | 1,0000 | 0,0714 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Hybrid | 3 | 0,6667 | 0,1429 | 1,0000 |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Hybrid | 6 | 0,8333 | 0,3571 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Sparse | 6 | 0,3333 | 0,1538 | 0,2000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Dense | 1 | 1,0000 | 0,0769 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Dense | 3 | 0,6667 | 0,1538 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Dense | 6 | 0,3333 | 0,1538 | 1,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Hybrid | 3 | 0,6667 | 0,1538 | 0,5000 |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Hybrid | 6 | 0,5000 | 0,2308 | 0,5000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Sparse | 6 | 0,1667 | 0,1111 | 0,2500 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Dense | 3 | 0,3333 | 0,1111 | 1,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Dense | 6 | 0,3333 | 0,2222 | 1,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Hybrid | 3 | 0,6667 | 0,2222 | 1,0000 |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Hybrid | 6 | 0,3333 | 0,2222 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Sparse | 3 | 0,3333 | 0,0769 | 0,5000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Sparse | 6 | 0,3333 | 0,1538 | 0,5000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Dense | 1 | 1,0000 | 0,0769 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Dense | 3 | 0,6667 | 0,1538 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Dense | 6 | 0,3333 | 0,1538 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Hybrid | 1 | 1,0000 | 0,0769 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Hybrid | 3 | 0,6667 | 0,1538 | 1,0000 |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Hybrid | 6 | 0,3333 | 0,1538 | 1,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Sparse | 6 | 0,3333 | 0,2500 | 0,2000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Hybrid | 3 | 0,3333 | 0,1250 | 0,5000 |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Hybrid | 6 | 0,1667 | 0,1250 | 0,5000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,1429 | 1,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 1,0000 | 0,4286 | 1,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,8333 | 0,7143 | 1,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 1,0000 | 0,1429 | 1,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,6667 | 0,2857 | 1,0000 |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,3333 | 0,2857 | 1,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 3 | 0,3333 | 0,0833 | 0,3333 |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 6 | 0,1667 | 0,0833 | 0,3333 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 1 | 1,0000 | 0,0714 | 1,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 3 | 0,3333 | 0,0714 | 1,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 6 | 0,3333 | 0,1429 | 1,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 6 | 0,1667 | 0,0714 | 0,2500 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Sparse | 3 | 0,3333 | 0,1000 | 0,5000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Sparse | 6 | 0,1667 | 0,1000 | 0,5000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Dense | 6 | 0,5000 | 0,3000 | 0,2500 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Hybrid | 6 | 0,1667 | 0,1000 | 0,2000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Sparse | 1 | 1,0000 | 0,0667 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Sparse | 3 | 0,3333 | 0,0667 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Sparse | 6 | 0,3333 | 0,1333 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Dense | 3 | 0,6667 | 0,1333 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Dense | 6 | 0,5000 | 0,2000 | 1,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Hybrid | 3 | 0,3333 | 0,0667 | 0,3333 |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Hybrid | 6 | 0,5000 | 0,2000 | 0,3333 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Sparse | 1 | 1,0000 | 0,1667 | 1,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Sparse | 3 | 1,0000 | 0,5000 | 1,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Sparse | 6 | 0,6667 | 0,6667 | 1,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Dense | 3 | 0,3333 | 0,1667 | 0,5000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Dense | 6 | 0,3333 | 0,3333 | 0,5000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Hybrid | 1 | 1,0000 | 0,1667 | 1,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Hybrid | 3 | 0,6667 | 0,3333 | 1,0000 |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Hybrid | 6 | 0,5000 | 0,5000 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Sparse | 1 | 1,0000 | 0,0556 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Sparse | 3 | 1,0000 | 0,1667 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Sparse | 6 | 1,0000 | 0,3333 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Dense | 1 | 1,0000 | 0,0556 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Dense | 3 | 1,0000 | 0,1667 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Dense | 6 | 1,0000 | 0,3333 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Hybrid | 1 | 1,0000 | 0,0556 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Hybrid | 3 | 1,0000 | 0,1667 | 1,0000 |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Hybrid | 6 | 1,0000 | 0,3333 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,3333 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,3333 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 1,0000 | 0,3333 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,6667 | 0,4444 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,6667 | 0,2222 | 1,0000 |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,5000 | 0,3333 | 1,0000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 6 | 0,1667 | 0,1000 | 0,2500 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 3 | 0,3333 | 0,1000 | 0,5000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 6 | 0,1667 | 0,1000 | 0,5000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 3 | 0,3333 | 0,1000 | 0,5000 |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 6 | 0,1667 | 0,1000 | 0,5000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 3 | 0,6667 | 0,1333 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 6 | 0,3333 | 0,1333 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 3 | 1,0000 | 0,2000 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 6 | 0,5000 | 0,2000 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 3 | 0,6667 | 0,1333 | 1,0000 |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 6 | 0,5000 | 0,2000 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Sparse | 6 | 0,3333 | 0,1333 | 0,2500 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Dense | 1 | 1,0000 | 0,0667 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Dense | 3 | 0,6667 | 0,1333 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Dense | 6 | 0,8333 | 0,3333 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Hybrid | 3 | 0,6667 | 0,1333 | 1,0000 |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Hybrid | 6 | 0,5000 | 0,2000 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Sparse | 1 | 1,0000 | 0,0769 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Sparse | 3 | 0,3333 | 0,0769 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Sparse | 6 | 0,3333 | 0,1538 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Dense | 1 | 1,0000 | 0,0769 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Dense | 3 | 0,6667 | 0,1538 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Dense | 6 | 0,3333 | 0,1538 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Hybrid | 1 | 1,0000 | 0,0769 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Hybrid | 3 | 0,6667 | 0,1538 | 1,0000 |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Hybrid | 6 | 0,5000 | 0,2308 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Sparse | 1 | 1,0000 | 0,1000 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Sparse | 3 | 0,3333 | 0,1000 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Sparse | 6 | 0,3333 | 0,2000 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Dense | 3 | 0,6667 | 0,2000 | 0,5000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Dense | 6 | 0,3333 | 0,2000 | 0,5000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Hybrid | 1 | 1,0000 | 0,1000 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Hybrid | 3 | 0,6667 | 0,2000 | 1,0000 |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Hybrid | 6 | 0,3333 | 0,2000 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Sparse | 1 | 1,0000 | 0,1667 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Sparse | 3 | 1,0000 | 0,5000 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Sparse | 6 | 0,6667 | 0,6667 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Dense | 3 | 0,3333 | 0,1667 | 0,3333 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Dense | 6 | 0,1667 | 0,1667 | 0,3333 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Hybrid | 1 | 1,0000 | 0,1667 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Hybrid | 3 | 0,6667 | 0,3333 | 1,0000 |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Hybrid | 6 | 0,5000 | 0,5000 | 1,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Sparse | 3 | 0,3333 | 0,0909 | 0,3333 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Sparse | 6 | 0,3333 | 0,1818 | 0,3333 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Dense | 3 | 0,3333 | 0,0909 | 0,3333 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Dense | 6 | 0,1667 | 0,0909 | 0,3333 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Hybrid | 3 | 0,3333 | 0,0909 | 0,5000 |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Hybrid | 6 | 0,3333 | 0,1818 | 0,5000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Dense | 1 | 1,0000 | 0,3333 | 1,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Dense | 3 | 0,3333 | 0,3333 | 1,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Dense | 6 | 0,1667 | 0,3333 | 1,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Hybrid | 6 | 0,1667 | 0,3333 | 0,1667 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Sparse | 3 | 0,3333 | 0,1667 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Sparse | 6 | 0,3333 | 0,3333 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Dense | 3 | 0,6667 | 0,3333 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Dense | 6 | 0,5000 | 0,5000 | 0,5000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Hybrid | 1 | 1,0000 | 0,1667 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Hybrid | 3 | 1,0000 | 0,5000 | 1,0000 |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Hybrid | 6 | 0,5000 | 0,5000 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Sparse | 1 | 1,0000 | 0,0625 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Sparse | 3 | 1,0000 | 0,1875 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Sparse | 6 | 0,6667 | 0,2500 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Dense | 1 | 1,0000 | 0,0625 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Dense | 3 | 1,0000 | 0,1875 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Dense | 6 | 0,8333 | 0,3125 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Hybrid | 1 | 1,0000 | 0,0625 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Hybrid | 3 | 1,0000 | 0,1875 | 1,0000 |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Hybrid | 6 | 0,8333 | 0,3125 | 1,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Sparse | 6 | 0,1667 | 0,2500 | 0,2000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Dense | 6 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Hybrid | 6 | 0,0000 | 0,0000 | 0,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,1667 | 0,2500 | 0,1667 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,6667 | 0,5000 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,5000 | 0,7500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 1,0000 | 0,2500 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,6667 | 0,5000 | 1,0000 |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,3333 | 0,5000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Sparse | 6 | 0,1667 | 0,2000 | 0,2500 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 1 | 1,0000 | 0,2000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 3 | 0,6667 | 0,4000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Dense | 6 | 0,6667 | 0,8000 | 1,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 3 | 0,3333 | 0,2000 | 0,3333 |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Hybrid | 6 | 0,5000 | 0,6000 | 0,3333 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 3 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Sparse | 6 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 3 | 0,6667 | 0,1250 | 0,5000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Dense | 6 | 0,3333 | 0,1250 | 0,5000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 3 | 0,3333 | 0,0625 | 0,3333 |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Hybrid | 6 | 0,3333 | 0,1250 | 0,3333 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 1 | 1,0000 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 3 | 0,3333 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Sparse | 6 | 0,1667 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 1 | 1,0000 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 3 | 0,6667 | 0,2222 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Dense | 6 | 0,5000 | 0,3333 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 1 | 1,0000 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 3 | 0,3333 | 0,1111 | 1,0000 |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Hybrid | 6 | 0,3333 | 0,2222 | 1,0000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Sparse | 3 | 0,3333 | 0,0833 | 0,5000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Sparse | 6 | 0,3333 | 0,1667 | 0,5000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Dense | 3 | 0,6667 | 0,1667 | 0,5000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Dense | 6 | 0,5000 | 0,2500 | 0,5000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Hybrid | 1 | 1,0000 | 0,0833 | 1,0000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Hybrid | 3 | 0,6667 | 0,1667 | 1,0000 |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Hybrid | 6 | 0,6667 | 0,3333 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Sparse | 1 | 1,0000 | 0,0667 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Sparse | 3 | 0,3333 | 0,0667 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Sparse | 6 | 0,3333 | 0,1333 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Dense | 3 | 0,3333 | 0,0667 | 0,5000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Dense | 6 | 0,6667 | 0,2667 | 0,5000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Hybrid | 1 | 1,0000 | 0,0667 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Hybrid | 3 | 0,6667 | 0,1333 | 1,0000 |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Hybrid | 6 | 0,3333 | 0,1333 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Sparse | 1 | 1,0000 | 0,0588 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Sparse | 3 | 0,6667 | 0,1176 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Sparse | 6 | 0,5000 | 0,1765 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Dense | 1 | 1,0000 | 0,0588 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Dense | 3 | 0,6667 | 0,1176 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Dense | 6 | 0,8333 | 0,2941 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Hybrid | 1 | 1,0000 | 0,0588 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Hybrid | 3 | 1,0000 | 0,1765 | 1,0000 |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Hybrid | 6 | 0,5000 | 0,1765 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Sparse | 1 | 1,0000 | 0,0114 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Sparse | 3 | 0,6667 | 0,0227 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Sparse | 6 | 0,3333 | 0,0227 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Dense | 1 | 1,0000 | 0,0114 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Dense | 3 | 1,0000 | 0,0341 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Dense | 6 | 1,0000 | 0,0682 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Hybrid | 1 | 1,0000 | 0,0114 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Hybrid | 3 | 1,0000 | 0,0341 | 1,0000 |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Hybrid | 6 | 0,8333 | 0,0568 | 1,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 3 | 0,3333 | 0,1429 | 0,5000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 6 | 0,1667 | 0,1429 | 0,5000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 6 | 0,1667 | 0,1429 | 0,1667 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 1 | 0,0000 | 0,0000 | 0,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 3 | 0,0000 | 0,0000 | 0,0000 |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 6 | 0,1667 | 0,1429 | 0,2500 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 1 | 0,0000 | 0,0000 | 0,0000 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 3 | 0,3333 | 0,1250 | 0,3333 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Sparse | 6 | 0,3333 | 0,2500 | 0,3333 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 1 | 0,0000 | 0,0000 | 0,0000 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 3 | 0,0000 | 0,0000 | 0,0000 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Dense | 6 | 0,3333 | 0,2500 | 0,2500 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 1 | 1,0000 | 0,1250 | 1,0000 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 3 | 1,0000 | 0,3750 | 1,0000 |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Hybrid | 6 | 0,5000 | 0,3750 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Sparse | 1 | 1,0000 | 0,0357 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Sparse | 3 | 1,0000 | 0,1071 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Sparse | 6 | 1,0000 | 0,2143 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Dense | 1 | 1,0000 | 0,0357 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Dense | 3 | 1,0000 | 0,1071 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Dense | 6 | 1,0000 | 0,2143 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Hybrid | 1 | 1,0000 | 0,0357 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Hybrid | 3 | 1,0000 | 0,1071 | 1,0000 |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Hybrid | 6 | 1,0000 | 0,2143 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Sparse | 1 | 1,0000 | 0,0526 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Sparse | 3 | 1,0000 | 0,1579 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Sparse | 6 | 0,8333 | 0,2632 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Dense | 1 | 1,0000 | 0,0526 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Dense | 3 | 0,6667 | 0,1053 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Dense | 6 | 0,6667 | 0,2105 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Hybrid | 1 | 1,0000 | 0,0526 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Hybrid | 3 | 0,6667 | 0,1053 | 1,0000 |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Hybrid | 6 | 0,8333 | 0,2632 | 1,0000 |

## Hasil Pengujian RAGAS

In [7]:
ragas_directory = PROJECT / "evaluation/results/ragas_64_konteks"
if ULANG_RAGAS:
    ragas_directory = HASIL / ("ragas_" + RAGAS_RUN_ID)

arguments = [
    "-m", "evaluation.run_ragas_complete",
    "--questions", DATASET / "questions.jsonl",
    "--ground-truth", DATASET / "ground_truth.jsonl",
    "--output-dir", ragas_directory,
    "--context-policy", "retrieved_top_k",
    "--embedding-cache", EMBEDDING_CACHE,
    "--evaluator-max-tokens", "8192",
]
if (ragas_directory / "checkpoint.json").is_file():
    arguments.append("--resume")
jalankan(*arguments, allow_incomplete=True)
ragas = read_json(ragas_directory / "checkpoint.json")
ringkasan_ragas, rincian_ragas, status_ragas = ragas_tables(questions, references, ragas, retrieval)
konteks_ragas = ragas_context_table(questions, ragas)
jumlah_tanpa_konteks = status_ragas.get("skipped_no_context", 0)
cakupan_ragas = 100 * status_ragas.get("scored", 0) / len(rincian_ragas)
cakupan_teks = f"{cakupan_ragas:.2f}".replace(".", ",")
protokol_teks = (
    "RAGAS menguji retrieval dan generasi dari enam konteks teratas Hybrid, "
    "tanpa ambang penolakan aplikasi."
)
print(
    f"Dataset: {len(rincian_ragas)} pertanyaan; {status_ragas.get('scored', 0)} dinilai. "
    f"Cakupan penilaian: {cakupan_teks}%."
)
display(Markdown(protokol_teks))
display(Markdown(markdown_table(ringkasan_ragas)))
status_evaluasi = ragas["status"]
if status_evaluasi != "completed":
    pesan_status = ragas.get("connection_check", {}).get("message", status_evaluasi)
    display(Markdown(f"**Status RAGAS: {pesan_status}** Skor yang belum tersedia ditampilkan dengan tanda strip."))


Dataset: 64 pertanyaan; 64 dinilai. Cakupan penilaian: 100,00%.


RAGAS menguji retrieval dan generasi dari enam konteks teratas Hybrid, tanpa ambang penolakan aplikasi.

| Metrik | Rata-rata | Pertanyaan dinilai |
| --- | --- | --- |
| Faithfulness | 0,7466 | 64 |
| Answer Relevancy | 0,7679 | 64 |
| Context Precision | 0,5620 | 64 |
| Context Recall | 0,8462 | 64 |

### Hasil Pengujian RAGAS per Pertanyaan

In [8]:
kolom_skor_ragas = [
    "No", "Emiten", "Pertanyaan", "Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall", "Status",
]
kolom_jawaban_ragas = ["No", "Emiten", "Pertanyaan", "Jawaban sistem", "Jawaban acuan", "Keterangan"]
tabel_skor_ragas = [{kolom: row[kolom] for kolom in kolom_skor_ragas} for row in rincian_ragas]
tabel_jawaban_ragas = [{kolom: row[kolom] for kolom in kolom_jawaban_ragas} for row in rincian_ragas]
display(Markdown("#### Skor RAGAS per Pertanyaan"))
display(Markdown(markdown_table(tabel_skor_ragas)))
display(Markdown("#### Jawaban Sistem dan Jawaban Acuan"))
display(Markdown(markdown_table(tabel_jawaban_ragas)))


#### Skor RAGAS per Pertanyaan

| No | Emiten | Pertanyaan | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Status |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | 1,0000 | 1,0000 | 0,4167 | 1,0000 | Dinilai |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | 1,0000 | 0,8986 | 0,3250 | 1,0000 | Dinilai |
| 3 | BACH | Kapan perusahaan BACH didirikan? | 1,0000 | 0,9900 | 1,0000 | 1,0000 | Dinilai |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | 1,0000 | 0,5627 | 0,3667 | 1,0000 | Dinilai |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | 0,0000 | 1,0000 | 0,4167 | 1,0000 | Dinilai |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,8863 | 0,5000 | 1,0000 | Dinilai |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | 1,0000 | 0,6153 | 0,8333 | 0,6667 | Dinilai |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | 1,0000 | 0,5267 | 1,0000 | 0,8000 | Dinilai |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | 0,0000 | 1,0000 | 1,0000 | 1,0000 | Dinilai |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | 0,5000 | 0,9547 | 0,9167 | 1,0000 | Dinilai |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | 0,0000 | 0,5623 | 0,5889 | 1,0000 | Dinilai |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | 1,0000 | 0,7083 | 0,6667 | 1,0000 | Dinilai |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,6455 | 0,2500 | 1,0000 | Dinilai |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,6879 | 0,4444 | 1,0000 | Dinilai |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | 0,6667 | 0,0000 | 0,0000 | 0,5000 | Dinilai |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | 0,5000 | 0,6246 | 0,0000 | 1,0000 | Dinilai |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | 1,0000 | 1,0000 | 0,5000 | 1,0000 | Dinilai |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | 1,0000 | 0,9464 | 0,7000 | 1,0000 | Dinilai |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | 1,0000 | 0,6035 | 1,0000 | 1,0000 | Dinilai |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | 0,7500 | 0,8128 | 1,0000 | 1,0000 | Dinilai |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 0,0000 | 0,9211 | 0,2000 | 1,0000 | Dinilai |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 0,0000 | 0,8659 | 0,4667 | 1,0000 | Dinilai |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | 0,2000 | 0,9010 | 0,5000 | 0,5000 | Dinilai |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | 0,5000 | 0,7433 | 0,0000 | 0,6667 | Dinilai |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | 1,0000 | 1,0000 | 0,8875 | 1,0000 | Dinilai |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | 0,0000 | 0,9553 | 0,6458 | 1,0000 | Dinilai |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | 0,0000 | 0,5695 | 0,8333 | 1,0000 | Dinilai |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | 1,0000 | 0,6737 | 1,0000 | 0,8571 | Dinilai |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | 1,0000 | 0,9618 | 0,5000 | 1,0000 | Dinilai |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | 0,0000 | 0,8637 | 1,0000 | 1,0000 | Dinilai |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | 1,0000 | 0,6660 | 0,0000 | 0,5000 | Dinilai |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | 0,5000 | 0,6434 | 0,0000 | 0,6667 | Dinilai |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | 1,0000 | 1,0000 | 0,2000 | 1,0000 | Dinilai |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | 1,0000 | 0,9451 | 0,4778 | 1,0000 | Dinilai |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | 0,0000 | 0,5248 | 0,1667 | 0,5000 | Dinilai |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | 1,0000 | 0,8257 | 0,8333 | 1,0000 | Dinilai |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | 1,0000 | 0,5491 | 0,5000 | 1,0000 | Dinilai |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,8095 | 0,7556 | 1,0000 | Dinilai |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | 1,0000 | 0,6661 | 0,0000 | 0,3333 | Dinilai |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | 0,5000 | 0,6699 | 1,0000 | 0,6667 | Dinilai |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | 1,0000 | 1,0000 | 0,7556 | 1,0000 | Dinilai |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | 1,0000 | 0,9615 | 0,8667 | 1,0000 | Dinilai |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | 1,0000 | 0,8853 | 0,8333 | 1,0000 | Dinilai |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | 1,0000 | 0,6441 | 0,8667 | 1,0000 | Dinilai |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | 1,0000 | 0,7812 | 0,0000 | 0,0000 | Dinilai |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | 1,0000 | 0,7130 | 0,5000 | 0,6667 | Dinilai |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | 1,0000 | 0,0000 | 0,1667 | 0,0000 | Dinilai |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | 1,0000 | 0,8389 | 0,3333 | 1,0000 | Dinilai |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | 1,0000 | 0,6222 | 1,0000 | 1,0000 | Dinilai |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | 1,0000 | 0,9557 | 0,9267 | 1,0000 | Dinilai |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | 0,5000 | 0,5972 | 0,0000 | 0,0000 | Dinilai |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | 0,0000 | 0,6179 | 0,0000 | 0,0000 | Dinilai |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,8831 | 1,0000 | 1,0000 | Dinilai |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | 1,0000 | 0,8944 | 0,4778 | 1,0000 | Dinilai |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | 1,0000 | 0,6772 | 0,0000 | 0,6667 | Dinilai |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | 0,5000 | 0,6857 | 0,0000 | 0,5000 | Dinilai |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | 1,0000 | 0,9315 | 0,8167 | 1,0000 | Dinilai |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | 1,0000 | 1,0000 | 0,9167 | 1,0000 | Dinilai |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | 0,0000 | 0,9373 | 1,0000 | 1,0000 | Dinilai |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | 1,0000 | 0,8611 | 1,0000 | 1,0000 | Dinilai |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | 1,0000 | 0,7430 | 0,2500 | 1,0000 | Dinilai |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | 1,0000 | 0,9602 | 1,0000 | 1,0000 | Dinilai |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | 0,8333 | 0,5651 | 0,6667 | 1,0000 | Dinilai |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | 0,8333 | 0,6072 | 0,7000 | 0,6667 | Dinilai |

#### Jawaban Sistem dan Jawaban Acuan

| No | Emiten | Pertanyaan | Jawaban sistem | Jawaban acuan | Keterangan |
| --- | --- | --- | --- | --- | --- |
| 1 | BACH | Berapa jumlah saham baru yang ditawarkan dalam IPO BACH? | Jumlah saham baru yang ditawarkan dalam IPO BACH adalah 615.000.000 (enam ratus lima belas juta) lembar saham. | sebanyak 615.000.000 (enam ratus lima belas juta) saham biasa atas nama yang merupakan saham baru  |  |
| 2 | BACH | Siapa penjamin pelaksana emisi efek dalam IPO BACH? | Penjamin Pelaksana Emisi Efek dalam IPO BACH adalah PT Erdikha Elit Sekuritas. | PT ERDIKHA ELIT SEKURITAS |  |
| 3 | BACH | Kapan perusahaan BACH didirikan? | Perusahaan BACH didirikan pada tanggal 4 September 2006. | Akta Pendirian Perseroan Terbatas No. 09 tanggal 4 September 2006  |  |
| 4 | BACH | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama BACH pada bagian Faktor Risiko? | Risiko usaha paling utama yang dihadapi Perseroan adalah risiko ketergantungan terhadap pihak pemasok genset. | Ketergantungan terhadap Pihak Pemasok Genset  |  |
| 5 | BACH | Berapa total liabilitas BACH pada 31 Desember 2025? | Total liabilitas BACH pada 31 Desember 2025 adalah Rp696.676.897.249. | Per 31 Desember 2025. JUMLAH LIABILITAS 696.676.897.249 (dalam Rupiah)  |  |
| 6 | BACH | Berapa pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Pendapatan neto BACH untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian adalah Rp1.732.944.872.499. | Untuk tahun yang berakhir pada 31 Desember 2025. PENDAPATAN NETO 1.732.944.872.499 (dalam Rupiah)  |  |
| 7 | BACH | Untuk apa dana hasil IPO BACH akan digunakan? | Dana hasil Penawaran Umum Perdana Saham digunakan untuk pembayaran sebagian utang kepada PT Bank Permata Tbk atas fasilitas pinjaman jangka panjang Omnibus Revolving Loan sebesar Rp91.020.000.000, dan sisa dana digunakan sebagai modal kerja untuk pembelian genset yang dijual maupun disewakan. | Sebesar Rp91.020.000.000 (sembilan puluh satu miliar dua puluh juta Rupiah) akan digunakan Perseroan untuk pembayaran sebagian utang kepada PT Bank Permata Tbk. atas fasilitas pinjaman jangka pendek yaitu Omnibus Revolving Loan Sisa dana Penawaran Umum Perdana Saham akan digunakan sebagai modal kerja, yaitu dalam rangka pembelian genset untuk dijual maupun disewakan |  |
| 8 | BACH | Bagaimana kebijakan pembagian dividen BACH? | Perseroan berniat melakukan pembayaran dividen kas (Rupiah) sebanyak-banyaknya 50% untuk masa yang akan datang, mulai tahun 2027 berdasarkan laba bersih tahun 2026,. Kebijakan ini merupakan pernyataan maksud saat ini dan tidak mengikat secara hukum, serta Direksi dapat mengubah jumlah dividen atau tidak membayarkannya sama sekali dengan persetujuan RUPS,. Keputusan pembagian dividen ditetapkan melalui persetujuan pemegang saham pada RUPS tahunan berdasarkan rekomendasi Direksi, dan Perseroan dapat membagikan dividen jika mencatatkan saldo laba positif. Pembagian dividen interim dapat dilakukan sebelum berakhirnya tahun keuangan sepanjang diperbolehkan Anggaran Dasar dan tidak menyebabkan aset bersih menjadi kurang dari modal ditempatkan dan disetor penuh serta cadangan wajib, setelah mendapat persetujuan Dewan Komisaris. Seluruh saham yang telah ditempatkan dan disetor penuh mempunyai hak yang sama dan sederajat atas pembagian dividen. Perseroan berkomitmen membentuk cadangan sesuai UUPT dan tidak memiliki negative covenants terkait pembatasan pihak ketiga dalam pembagian dividen. | Setelah dilaksanakannya Penawaran Umum Perdana Saham, Perseroan berniat untuk melakukan pembayaran dividen kas (Rupiah) sebanyak-banyaknya 50% (lima puluh persen) untuk masa yang akan datang, mulai tahun 2027 berdasarkan dari laba bersih tahun 2026. Kebijakan dividen Perseroan adalah suatu pernyataan dari maksud saat ini dan tidak mengikat secara hukum, dan dengan tetap memperhatikan persetujuan Rapat Umum Pemegang Saham Perseroan, Direksi Perseroan dapat, dari waktu ke waktu, mengubah kebijakan pembagian dividen Perseroan. Dalam kebijakannya, Direksi Perseroan dapat mengubah jumlah dividen yang akan dibayarkan atau tidak melakukan pembayaran dividen sama sekali. Berdasarkan UUPT, pembagian dividen dilakukan berdasarkan keputusan Rapat Umum Pemegang Saham Tahunan (RUPS) atau Rapat Umum Pemegang Saham Luar Biasa (RUPSLB). Sebelum berakhirnya tahun keuangan, dividen interim dapat dibagikan sepanjang hal itu diperbolehkan oleh Anggaran Dasar Perseroan dan pembagian dividen interim tidak menyebabkan aset bersih Perseroan menjadi kurang dari modal ditempatkan dan disetor penuh dan cadangan wajib Perseroan. Pembagian dividen interim tersebut ditetapkan oleh Direksi setelah mendapat persetujuan dari Dewan Komisaris. Sampai dengan Prospektus ini diterbitkan, tidak terdapat pembatasan-pembatasan (negative covenant) dalam hal Perseroan mengumumkan dan membagikan dividen kepada pemegang saham Perseroan dalam perjanjian pinjaman antara Perseroan dengan para krediturnya. |  |
| 9 | EMMI | Berapa jumlah saham baru yang ditawarkan dalam IPO EMMI? | Jumlah saham baru yang ditawarkan dalam IPO EMMI adalah 522.857.000 saham. | Sebesar 522.857.000 (lima ratus dua puluh dua juta delapan ratus lima puluh tujuh ribu) saham biasa atas nama yang seluruhnya adalah Saham Baru |  |
| 10 | EMMI | Siapa penjamin pelaksana emisi efek dalam IPO EMMI? | Penjamin Pelaksana Emisi Efek dalam IPO EMMI adalah PT BRI Danareksa Sekuritas dan PT INA Sekuritas Indonesia. | PT BRI DANAREKSA SEKURITAS PT INA SEKURITAS INDONESIA |  |
| 11 | EMMI | Apa kegiatan usaha utama yang dijalankan EMMI untuk memperoleh pendapatan? | Kegiatan usaha utama Perseroan adalah perdagangan besar alat laboratorium, alat farmasi, dan alat kedokteran untuk manusia. | Perseroan merupakan suatu perusahaan terbatas yang memiliki kegiatan usaha utama di bidang perdagangan besar alat laboratorium, alat farmasi dan alat kedokteran untuk manusia (Kode KBLI 46691), yang mencakup kegiatan usaha perdagangan besar alat laboratorium, alat farmasi, dan alat kedokteran untuk manusia. |  |
| 12 | EMMI | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama EMMI pada bagian Faktor Risiko? | Risiko usaha paling utama PT EMMI adalah risiko konsentrasi pelanggan. | Risiko Konsentrasi Pelanggan |  |
| 13 | EMMI | Berapa total aset EMMI pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Total aset PT Esa Medika Mandiri Tbk pada 31 Desember 2025 sebesar Rp564.439.161.798. | Per 31 Desember 2025. TOTAL ASET 564.439.161.798 (disajikan dalam Rupiah) |  |
| 14 | EMMI | Berapa penjualan bersih EMMI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Penjualan bersih PT Esa Medika Mandiri Tbk untuk tahun yang berakhir pada 31 Desember 2025 sebesar Rp454.636.079.798. | Untuk tahun yang berakhir pada 31 Desember 2025. PENJUALAN BERSIH 454.636.079.798 (disajikan dalam Rupiah) |  |
| 15 | EMMI | Apa kegiatan usaha utama EMMI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Kegiatan usaha utama EMMI tidak ditemukan dalam sumber yang tersedia. Satu alokasi penggunaan dana IPO adalah sebesar Rp50.000.000.000,- (lima puluh miliar Rupiah) untuk pembayaran sebagian pokok pinjaman Perseroan. | Perseroan merupakan suatu perusahaan terbatas yang memiliki kegiatan usaha utama di bidang perdagangan besar alat laboratorium, alat farmasi dan alat kedokteran untuk manusia (Kode KBLI 46691), yang mencakup kegiatan usaha perdagangan besar alat laboratorium, alat farmasi, dan alat kedokteran untuk manusia.Sebesar Rp50.000.000.000,- (lima puluh miliar Rupiah) akan digunakan untuk pembayaran sebagian pokok pinjaman Perseroan. |  |
| 16 | EMMI | Apa risiko usaha paling utama EMMI menurut bagian Faktor Risiko? Sebutkan juga penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Risiko usaha paling utama EMMI menurut bagian Faktor Risiko adalah Risiko Konsentrasi Pelanggan. Penjualan bersih konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 adalah Rp454.636.079.798. | Untuk tahun yang berakhir pada 31 Desember 2025. Risiko Konsentrasi Pelanggan PENJUALAN BERSIH 454.636.079.798 (disajikan dalam Rupiah) |  |
| 17 | JECX | Berapa jumlah saham baru yang ditawarkan dalam IPO JECX? | Jumlah saham baru yang ditawarkan dalam IPO JECX adalah 325.322.300 (tiga ratus dua puluh lima juta tiga ratus dua puluh dua ribu tiga ratus) saham biasa. | 325.322.300 (tiga ratus dua puluh lima juta tiga ratus dua puluh dua ribu tiga ratus) saham baru yang dikeluarkan dari portepel Perseroan  |  |
| 18 | JECX | Siapa penjamin pelaksana emisi efek dalam IPO JECX? | Penjamin Pelaksana Emisi Efek dalam IPO JECX adalah PT Trimegah Sekuritas Indonesia Tbk. | PT TRIMEGAH SEKURITAS INDONESIA Tbk |  |
| 19 | JECX | Di mana alamat kantor pusat perusahaan JECX? | Alamat kantor pusat Perseroan adalah Jl. Teuku Cik Ditiro No. 46, Menteng, Jakarta Pusat, 10310 - Indonesia. | Jl. Teuku Cik Ditiro No. 46, Menteng Jakarta Pusat, 10310 - Indonesia  |  |
| 20 | JECX | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama JECX pada bagian Faktor Risiko? | Risiko usaha paling utama JECX adalah ketergantungan Grup Perseroan terhadap para dokter, perawat, dan tenaga profesional yang berperan penting dalam perusahaan dalam melaksanakan strategi pertumbuhan Grup Perseroan maupun kelancaran operasional Grup Perseroan. | Ketergantungan Grup Perseroan terhadap para dokter, perawat, dan tenaga profesional yang berperan penting dalam perusahaan dalam melaksanakan strategi pertumbuhan Grup Perseroan maupun kelancaran operasional Grup Perseroan |  |
| 21 | JECX | Berapa total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Total aset JECX pada 31 Desember 2025 berdasarkan laporan posisi keuangan konsolidasian adalah Rp271.415 juta. | Per 31 Desember 2025. Jumlah Aset 1.451.329 (dalam jutaan Rupiah) |  |
| 22 | JECX | Berapa pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Pendapatan JECX untuk tahun yang berakhir pada 31 Desember 2025 adalah Rp926.763 juta. | Untuk tahun yang berakhir pada 31 Desember 2025. Pendapatan 926.763 (dalam jutaan Rupiah) |  |
| 23 | JECX | Untuk apa dana hasil IPO JECX akan digunakan? | Dana hasil Penawaran Umum Perdana Saham JECX akan digunakan untuk: 1) pembayaran lebih awal kepada PT Bank Central Asia Tbk sebesar Rp40.000.000.000; 2) pembayaran lebih awal kepada PT Bank HSBC Indonesia sebesar Rp100.000.000.000; 3) pemberian pinjaman kepada perusahaan terkendali Perseroan (JCS) yang selanjutnya digunakan untuk membayar sebagian pokok utang kepada Bank BCA; dan 4) sisanya untuk modal kerja Perseroan berupa biaya operasional sehari-hari seperti gaji dan tunjangan karyawan. | Sebesar Rp40.000.000.000 (empat puluh miliar Rupiah) akan digunakan Perseroan untuk pembayaran lebih awal kepada PT Bank Central Asia Tbk (“Bank BCA”) atas sebagian pokok pinjaman Perseroan. Sebesar Rp100.000.000.000 (seratus miliar Rupiah) akan digunakan Perseroan untuk pembayaran lebih awal kepada PT Bank HSBC Indonesia (“HSBC”) atas sebagian pokok pinjaman Perseroan. Sekitar Rp50.000.000.000 (lima puluh miliar Rupiah) akan disalurkan oleh Perseroan dalam bentuk setoran modal kepada NSB yang selanjutnya akan digunakan untuk modal kerja, termasuk namun tidak terbatas pada pembayaran sewa lahan dan gaji karyawan. Sebesar Rp100.000.000.000 (seratus miliar Rupiah) akan digunakan Perseroan untuk memberikan pinjaman kepada Orbita, perusahaan terkendali Perseroan yang selanjutnya akan digunakan Orbita untuk membayar sebagian pokok utang kepada Bank BCA. Sebesar Rp35.000.000.000 (tiga puluh lima miliar Rupiah) akan digunakan Perseroan untuk memberikan pinjaman kepada JCS, perusahaan terkendali Perseroan, yang selanjutnya akan digunakan JCS untuk membayar sebagian pokok utang kepada Bank BCA. Sisanya akan digunakan untuk modal kerja Perseroan yaitu biaya-biaya sehubungan dengan kegiatan operasional Perseroan sehari-hari, antara lain biaya gaji dan tunjangan karyawan yang akan digunakan secara bertahap selambat-lambatnya sampai dengan 31 Desember 2027. |  |
| 24 | JECX | Apa risiko usaha paling utama JECX menurut bagian Faktor Risiko? Sebutkan juga pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Informasi mengenai risiko usaha paling utama JECX menurut bagian Faktor Risiko tidak tersedia di dokumen prospektus ini. Pendapatan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 adalah Rp926.763 juta. | Untuk tahun yang berakhir pada 31 Desember 2025. Ketergantungan Grup Perseroan terhadap para dokter, perawat, dan tenaga profesional yang berperan penting dalam perusahaan dalam melaksanakan strategi pertumbuhan Grup Perseroan maupun kelancaran operasional Grup Perseroan Pendapatan 926.763 (dalam jutaan Rupiah) |  |
| 25 | JELI | Berapa jumlah saham baru yang ditawarkan dalam IPO JELI? | Jumlah saham baru yang ditawarkan dalam IPO JELI adalah 266.000.000 (dua ratus enam puluh enam juta) saham biasa atas nama. | Sebesar 266.000.000 (dua ratus enam puluh enam juta) saham biasa atas nama yang seluruhnya terdiri dari Saham Baru |  |
| 26 | JELI | Siapa penjamin pelaksana emisi efek dalam IPO JELI? | Penjamin Pelaksana Emisi Efek dalam IPO JELI adalah PT Sucor Sekuritas. | PT SUCOR SEKURITAS |  |
| 27 | JELI | Apa kegiatan usaha utama yang dijalankan JELI untuk memperoleh pendapatan? | Kegiatan usaha utama yang dijalankan oleh Perseroan adalah industri makanan dan minuman penutup. | Bergerak dalam bidang Industri Makanan dan Minuman Penutup |  |
| 28 | JELI | Apa produk utama yang dihasilkan perusahaan JELI? | Produk utama yang dihasilkan Perseroan meliputi mini jelly, nata de coco, pudding, confectionery, snack, serta minuman siap minum (RTD). Selain itu, produk-produk utama lainnya mencakup aloe vera dan mini pudding. | Mini Jelly • Nata De Coco • Aloe vera • Pudding dan Mini Pudding • Confectionery • RTD dan RTD Pouch • Snack |  |
| 29 | JELI | Berapa total liabilitas JELI pada 31 Desember 2025? | Total liabilitas Perseroan (JELI) pada 31 Desember 2025 adalah Rp43.258.612.863. | Per 31 Desember 2025. Jumlah Liabilitas 406.594.316.759 (dalam Rupiah)  |  |
| 30 | JELI | Berapa penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi konsolidasian? Sebutkan nilai dan satuannya sesuai tabel. | Penjualan JELI untuk tahun yang berakhir pada 31 Desember 2025 sebesar Rp753.050.908.792. | Untuk tahun yang berakhir pada 31 Desember 2025. Penjualan 753.050.908.792 (dalam Rupiah)  |  |
| 31 | JELI | Apa kegiatan usaha utama JELI menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Kegiatan usaha utama JELI tidak ditemukan dalam sumber yang tersedia. Alokasi penggunaan dana IPO adalah sekitar 22,36% untuk modal kerja. | Bergerak dalam bidang Industri Makanan dan Minuman Penutup Sisanya sekitar 22,36% akan digunakan oleh Perseroan sebagai modal kerja, antara lain meliputi pembelian bahan baku, pembayaran biaya operasional, serta kegiatan pemasaran, guna mendukung peningkatan aktivitas operasional dan pertumbuhan usaha Perseroan. |  |
| 32 | JELI | Apa risiko usaha paling utama JELI menurut bagian Faktor Risiko? Sebutkan juga penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Informasi mengenai risiko usaha paling utama JELI menurut bagian Faktor Risiko tidak tersedia di dokumen prospektus ini. Penjualan konsolidasian untuk tahun yang berakhir pada 31 Desember 2025 adalah Rp753.050.908.792. | Untuk tahun yang berakhir pada 31 Desember 2025. Risiko Terkait dengan Ketersediaan dan Volatilitas Harga Bahan Baku Penjualan 753.050.908.792 (dalam Rupiah) |  |
| 33 | PRDL | Berapa harga penawaran per saham dalam Penawaran Umum Perdana Saham PRDL? | Harga Penawaran per saham dalam Penawaran Umum Perdana Saham PRDL adalah Rp120,- (seratus dua puluh Rupiah). | Harga Penawaran sebesar Rp120,- (seratus dua puluh Rupiah) setiap saham |  |
| 34 | PRDL | Siapa penjamin pelaksana emisi efek dalam IPO PRDL? | Penjamin Pelaksana Emisi Efek dalam IPO PRDL adalah PT Sucor Sekuritas. | PT SUCOR SEKURITAS |  |
| 35 | PRDL | Apa kegiatan usaha utama yang dijalankan PRDL untuk memperoleh pendapatan? | Kegiatan usaha utama yang dijalankan Perseroan adalah Aktivitas Pelayanan Penunjang Kesehatan (KBLI No. 86903) dan Aktivitas Klinik Swasta (KBLI No. 86105). | Industri Alat Kesehatan dalam Subgolongan 2101. Kelompok ini mencakup usaha pembuatan dan pengolahan alat Kesehatan terkait diagnosa medis dan produk lainnya dalam subgolongan 2011. |  |
| 36 | PRDL | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama PRDL pada bagian Faktor Risiko? | Risiko usaha paling utama PRDL adalah risiko ketergantungan terhadap belanja Pemerintah pada sektor kesehatan. | Risiko Ketergantungan terhadap Belanja Pemerintah pada Sektor Kesehatan |  |
| 37 | PRDL | Bagaimana kebijakan pembagian dividen PRDL? | Setelah Penawaran Umum Perdana Saham, Perseroan akan membayarkan dividen tunai kepada pemegang saham dengan rasio sebanyak-banyaknya 20% dari laba bersih setelah penyisihan untuk cadangan wajib mulai tahun buku 2026. Pembagian dividen ditentukan berdasarkan hasil RUPS Tahunan, kinerja, dan rencana pengembangan bisnis Perseroan, serta tidak ada pembatasan yang membatasi hak pemegang saham publik dalam menerima dividen. Pada tahun 2026, Perseroan juga melakukan pembagian dividen dalam bentuk tambahan penyertaan modal saham sebesar Rp46.000.000.000,- yang berasal dari saldo laba kumulatif tahun 2024. | Setelah Penawaran Umum Perdana Saham, Perseroan untuk setiap tahunnya berencana membayarkan dividen tunai kepada pemegang saham Perseroan dengan rasio sebanyak-banyaknya 20% (dua puluh persen) dari laba bersih setelah penyisihan untuk cadangan wajib yang dimulai dari tahun buku 2026 dengan tidak mengabaikan tingkat kesehatan Perseroan dan tanpa mengurangi hak dari Rapat Umum Pemegang Saham Perseroan untuk menentukan lain sesuai dengan Anggaran Dasar Perseroan. Pembagian dividen oleh Perseroan ditentukan berdasarkan hasil RUPS Tahunan Perseroan dan juga kinerja serta rencana pengembangan bisnis Perseroan. Tidak terdapat pembatasan-pembatasan yang dapat membatasi hak pemegang saham publik dalam menerima dividen. Pada tahun 2026, Perseroan melakukan pembagian dividen dalam bentuk tambahan penyertaan modal saham sebesar Rp46.000.000.000,- (empat puluh enam miliar Rupiah) yang berasal dari saldo laba kumulatif tahun 2024. |  |
| 38 | PRDL | Berapa pendapatan dari kontrak dengan pelanggan Perseroan PRDL untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 sebesar Rp74.371.107 ribu. | Untuk tahun yang berakhir pada 31 Desember 2025. Pendapatan dari kontrak dengan pelanggan 74.371.107 (dalam ribuan Rupiah) |  |
| 39 | PRDL | Apa kegiatan usaha utama PRDL menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Kegiatan usaha utama PRDL tidak ditemukan dalam sumber yang tersedia. Satu alokasi penggunaan dana IPO adalah sekitar 10,18% untuk modal kerja Perseroan termasuk pembelian bahan baku, biaya product & development dan selling & marketing. | Industri Alat Kesehatan dalam Subgolongan 2101. Kelompok ini mencakup usaha pembuatan dan pengolahan alat Kesehatan terkait diagnosa medis dan produk lainnya dalam subgolongan 2011. Sisanya sekitar 10,18% (sepuluh koma satu delapan persen) akan digunakan untuk modal kerja (working capital) Perseroan termasuk namun tidak terbatas pada pembelian bahan baku, biaya product & development dan selling & marketing |  |
| 40 | PRDL | Apa risiko usaha paling utama PRDL menurut bagian Faktor Risiko? Sebutkan juga pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Informasi mengenai risiko usaha paling utama menurut bagian Faktor Risiko tidak tersedia di dokumen prospektus ini. Pendapatan dari kontrak dengan pelanggan Perseroan untuk tahun yang berakhir pada 31 Desember 2025 adalah Rp74.371.107 ribu. | Untuk tahun yang berakhir pada 31 Desember 2025. Risiko Ketergantungan terhadap Belanja Pemerintah pada Sektor Kesehatan. Pendapatan dari kontrak dengan pelanggan 74.371.107 (dalam ribuan Rupiah)  |  |
| 41 | RANS | Berapa jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik? | Jumlah saham yang ditawarkan saat melakukan penawaran saham kepada publik adalah 2.525.000.000,00 (dua miliar lima ratus dua puluh lima juta) Saham biasa atas nama. | Sebanyak 2.525.000.000 (dua miliar lima ratus dua puluh lima juta) saham biasa atas nama |  |
| 42 | RANS | Siapa penjamin pelaksana emisi efek dalam IPO RANS? | Penjamin Pelaksana Emisi Efek dalam IPO RANS adalah PT Trimegah Sekuritas Indonesia Tbk. | PT TRIMEGAH SEKURITAS INDONESIA Tbk |  |
| 43 | RANS | Siapa pemegang saham pengendali perusahaan? | Pemegang saham pengendali Perseroan adalah Raffi Farid Ahmad. | Raffi Farid Ahmad, selaku Pengendali Perseroan |  |
| 44 | RANS | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama RANS pada bagian Faktor Risiko? | Risiko usaha paling utama Perseroan adalah risiko ketergantungan terhadap talenta utama, yaitu Raffi Ahmad dan Nagita Slavina beserta keluarga. | Risiko Ketergantungan terhadap Talent Utama, yaitu Raffi Ahmad dan Nagita Slavina beserta Keluarga |  |
| 45 | RANS | Bagaimana tren pertumbuhan industri tempat perusahaan beroperasi? | Industri tempat Perseroan beroperasi, yaitu media, hiburan, event, dan produk konsumen, memiliki tingkat persaingan relatif tinggi dan terus berkembang. | Di dalam kanal digital, pertumbuhan belanja social media advertising dan influencer marketing juga meningkat secara signifikan (masing-masing +11,3% dan +14,4%). Konsisten dengan tren tersebut, proyeksi pihak ketiga memperkirakan pasar digital advertising Indonesia tumbuh dari US$3,23 miliar (2025) menjadi US$3,41 miliar (2026). |  |
| 46 | RANS | Berapa dana IPO yang dialokasikan untuk pembelian aset? | Dana IPO yang dialokasikan untuk belanja modal (pembelian aset) pembangunan wahana "Cipungland" adalah sekitar Rp80.000.000.000,-. | Sekitar Rp80.000.000.000,- (delapan puluh miliar Rupiah) akan digunakan oleh Perseroan untuk belanja modal (capital expenditure) atas rencana ekspansi usaha dalam rangka pembangunan wahana bermain dan belajar edukatif dengan nama “Cipungland”, yang akan dilaksanakan secara bertahap pada tahun 2026–2028. Adapun alokasi dana hasil Penawaran Umum untuk Cipungland ini terdiri atas: (i) belanja modal pembangunan wahana Cipungland untuk 9 titik sekitar 45,18%; dan (ii) kebutuhan sewa lokasi untuk 7 titik sekitar 54,82%, yang mencakup biaya sewa untuk 3 tahun. |  |
| 47 | RANS | Apa kegiatan usaha utama RANS menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Kegiatan usaha utama Perseroan tidak disebutkan secara eksplisit dalam kutipan yang tersedia, namun alokasi penggunaan dana IPO mencakup sekitar 21,25% untuk ekspansi usaha melalui akuisisi kepemilikan saham pada PT Rans Kosmetika Indonesia dan sekitar 6,75% untuk penyetoran modal kepada Entitas Anak Perseroan (RNS). | Perseroan bergerak dalam bidang usaha media dan hiburan, dalam hal ini produksi dan distribusi konten, pengelolaan intellectual property (IP), penyelenggaraan event dan seni pertunjukan, periklanan, aktivitas taman bertema dan taman hiburan, arena permainan, serta aktivitas perusahaan holding. Sekitar Rp85.000.000.000,- (delapan puluh lima miliar Rupiah) akan digunakan oleh Perseroan untuk ekspansi usaha melalui akuisisi kepemilikan saham pada PT Rans Kosmetika Indonesia (“Slavina”). |  |
| 48 | RANS | Berapa jumlah saham perusahaan setelah IPO? | Jumlah saham Perseroan setelah Penawaran Umum Perdana Saham adalah 12.609.250.000 (dua belas miliar enam ratus sembilan juta dua ratus lima puluh ribu) saham. | jumlah saham yang akan dicatatkan oleh Perseroan di Bursa Efek Indonesia seluruhnya adalah sebanyak 12.609.250.000 |  |
| 49 | SWAP | Berapa harga penawaran per saham yang dicantumkan dalam prospektus IPO SWAP? Jika masih berupa rentang, sebutkan batas bawah dan batas atasnya. | Harga penawaran per saham yang ditetapkan sebesar Rp ● setiap saham, dengan rentang harga yang dimasukkan oleh calon investor dalam Penawaran Awal adalah Rp130,- (batas bawah) sampai dengan Rp140,- (batas atas) setiap saham. | Harga Penawaran sebesar Rp130,- (seratus tiga puluh Rupiah) sampai dengan Rp140,- (seratus empat puluh Rupiah) setiap saham. |  |
| 50 | SWAP | Siapa penjamin pelaksana emisi efek dalam IPO SWAP? | Penjamin Pelaksana Emisi Efek dalam IPO SWAP adalah PT Sukadana Prima Sekuritas. | PT SUKADANA PRIMA SEKURITAS |  |
| 51 | SWAP | Apa kegiatan usaha utama yang dijalankan SWAP untuk memperoleh pendapatan? | Kegiatan usaha utama Perseroan adalah penjualan dan distribusi produk, khususnya produk Insahunt dan TopClean RSF Handrub. | Namun kegiatan usaha Perseroan yang saat ini telah benar-benar dijalankan saat ini adalah bergerak dalam bidang industri manufaktur produk kesehatan berbasis riset. |  |
| 52 | SWAP | Ketergantungan atau kondisi apa yang disebut sebagai risiko usaha paling utama SWAP pada bagian Faktor Risiko? | Risiko usaha paling utama bagi Perseroan adalah risiko fluktuasi harga saham Perseroan. | Risiko Distribusi dan Pemasaran |  |
| 53 | SWAP | Berapa total aset Perseroan SWAP pada 31 Desember 2025 berdasarkan laporan posisi keuangan? Sebutkan nilai dan satuannya sesuai tabel. | Total aset Perseroan SWAP pada 31 Desember 2025 adalah sebesar Rp33.362.492.250. | Total Aset Perseroan untuk tahun yang berakhir pada tanggal 31 Desember 2025 adalah sebesar Rp33.362.492.250 |  |
| 54 | SWAP | Berapa pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 berdasarkan laporan laba rugi? Sebutkan nilai dan satuannya sesuai tabel. | Pendapatan usaha Perseroan SWAP untuk tahun yang berakhir pada 31 Desember 2025 adalah sebesar Rp7.121.355.527. | Pendapatan Perseroan untuk tahun yang berakhir pada tanggal 31 Desember 2025 adalah sebesar Rp7.121.355.527 |  |
| 55 | SWAP | Apa kegiatan usaha utama SWAP menurut bagian Kegiatan Usaha? Sebutkan juga satu alokasi penggunaan dana IPO beserta nominal atau persentasenya menurut bagian Rencana Penggunaan Dana. | Kegiatan usaha utama SWAP adalah Industri Sabun Dan Bahan Pembersih Keperluan Rumah Tangga (20231). Salah satu alokasi penggunaan dana IPO adalah sekitar Rp500.000.000,- untuk peningkatan Sumber Daya Manusia (SDM) mencakup pelatihan SDM serta penambahan jumlah SDM untuk produk baru dan pengembangan kapasitas. | Industri Sabun Dan Bahan Pembersih Keperluan Rumah Tangga (20231). Namun kegiatan usaha Perseroan yang saat ini telah benar-benar dijalankan saat ini adalah bergerak dalam bidang industri manufaktur produk kesehatan berbasis riset. Sekitar Rp500.000.000,- (lima ratus juta Rupiah) akan digunakan untuk aktifitas peningkatan Sumber Daya Manusia (SDM) mencakup pelatihan SDM serta penambahan jumlah SDM untuk produk baru dan pengembangan kapasitas. |  |
| 56 | SWAP | Apa risiko usaha paling utama SWAP menurut bagian Faktor Risiko? Sebutkan juga pendapatan usaha Perseroan untuk tahun yang berakhir pada 31 Desember 2025 beserta satuannya. | Informasi mengenai risiko usaha paling utama SWAP menurut bagian Faktor Risiko tidak tersedia di dokumen prospektus ini. Pendapatan usaha Perseroan untuk tahun yang berakhir pada tanggal 31 Desember 2025 adalah sebesar Rp7.121.355.527. | Risiko Distribusi dan Pemasaran Pendapatan Perseroan untuk tahun yang berakhir pada tanggal 31 Desember 2025 adalah sebesar Rp7.121.355.527 |  |
| 57 | WBSA | Berapa harga penawaran saham WBSA pada prospektus ini? | Harga penawaran saham WBSA adalah Rp168,- (seratus enam puluh delapan Rupiah) per saham. | Harga Penawaran sebesar Rp168,- (seratus enam puluh delapan Rupiah) setiap saham |  |
| 58 | WBSA | Berapa jumlah saham baru yang ditawarkan dalam IPO WBSA? | Jumlah saham baru yang ditawarkan dalam IPO WBSA adalah 1.800.000.000 (satu miliar delapan ratus juta) saham biasa atas nama. | Penawaran Umum Perdana Saham sebesar 1.800.000.000 (satu miliar delapan ratus juta) saham biasa atas nama |  |
| 59 | WBSA | Siapa penjamin pelaksana emisi efek IPO WBSA? | Penjamin Pelaksana Emisi Efek IPO WBSA adalah PT OCBC Sekuritas Indonesia dan PT Semesta Indovest Sekuritas. | PT OCBC Sekuritas Indonesia PT Semesta Indovest Sekuritas |  |
| 60 | WBSA | Apa kegiatan usaha utama Perseroan? | Kegiatan usaha utama Perseroan adalah angkutan multimoda, angkutan bermotor untuk barang umum, angkutan bermotor untuk barang khusus, pergudangan dan penyimpanan, aktivitas cold storage, dan aktivitas perusahaan holding. | Saat ini, kegiatan usaha yang benar-benar dijalankan Perseroan adalah (i) angkutan multimoda; (ii) angkutan bermotor untuk barang umum; (iii) angkutan bermotor untuk barang khusus; (iv) pergudangan dan penyimpanan; (v) aktivitas cold storage dan (vi) aktivitas perusahaan holding. |  |
| 61 | WBSA | Berapa total aset WBSA pada 30 September 2025? Sebutkan satuannya. | Total aset Perseroan pada 30 September 2025 sebesar Rp1.157.015.301 ribu. | Per 30 September 2025. TOTAL ASET 1.157.015.301 (dalam ribuan Rupiah) |  |
| 62 | WBSA | Berapa total liabilitas WBSA pada 30 September 2025? Sebutkan satuannya. | Total liabilitas WBSA pada 30 September 2025 adalah Rp826.777.488 ribu. | Per 30 September 2025. TOTAL LIABILITAS 826.777.488 (dalam ribuan Rupiah) |  |
| 63 | WBSA | Bagaimana kebijakan pembagian dividen WBSA? | Pembagian dividen Perseroan ditentukan berdasarkan hasil RUPS Tahunan, kinerja, dan rencana pengembangan bisnis Perseroan. Rencana pembagian dividen sebanyak-banyaknya 10% dari laba bersih tahun berjalan setelah pajak yang berakhir pada tahun 2027 didasarkan pada kondisi Perseroan yang masih berada dalam fase pertumbuhan dengan prioritas arus kas untuk pengembangan usaha. Perseroan dapat mengusulkan dividen lebih tinggi dari yang diungkapkan dalam Prospektus dan membagikan dividen interim sebelum tahun buku berakhir dengan persetujuan Dewan Komisaris, selama kekayaan bersih tidak menjadi lebih kecil daripada modal ditempatkan dan disetor ditambah cadangan wajib serta tidak mengganggu kewajiban kepada kreditur. Pada Oktober 2025, Perseroan melakukan pembagian dividen dalam bentuk saham sebesar Rp30.000.000.000,- melalui kapitalisasi saldo laba. | Setelah Penawaran Umum Perdana Saham, Perseroan berencana untuk membayarkan dividen kas kepada Pemegang Saham Perseroan dengan rasio sebanyak-banyaknya 10% (sepuluh persen) dari laba bersih tahun buku yang bersangkutan, dimulai dari tahun buku 2027, dengan tidak mengabaikan tingkat kesehatan Perseroan dan tanpa mengurangi hak dari RUPS Perseroan untuk menentukan lain sesuai dengan Anggaran Dasar Perseroan. Pertimbangan Perseroan dalam menetapkan rencana pembagian dividen sebanyak-banyaknya 10% (sepuluh persen) dari laba bersih tahun berjalan setelah pajak yang berakhir pada tahun 2027 didasarkan pada kondisi Perseroan masih berada dalam fase pertumbuhan. Pada tahap ini, arus kas yang dihasilkan Perseroan akan diprioritaskan untuk mendukung kegiatan pengembangan usaha, termasuk namun tidak terbatas pada perluasan jaringan logistik, penguatan operasional, dan pengembangan layanan di masa yang akan datang. Pembagian dividen oleh Perseroan ditentukan berdasarkan hasil RUPS Tahunan Perseroan dan juga kinerja serta rencana pengembangan bisnis Perseroan. Perseroan dapat mempertimbangkan kemungkinan untuk mengusulkan pembagian dividen yang lebih tinggi dari yang telah diungkapkan dalam Prospektus dan juga dapat membagikan dividen interim sebelum tahun buku Perseroan berakhir sesuai dengan peraturan perundang-undangan yang berlaku dengan persetujuan terlebih dahulu dari Dewan Komisaris. Pada Oktober 2025, Perseroan melakukan pembagian dividen dalam bentuk saham dengan menerbitkan 30.000 saham baru Perseroan atau sebesar Rp30.000.000.000,- (tiga puluh miliar Rupiah) yang berasal dari kapitalisasi saldo laba Perseroan. |  |
| 64 | WBSA | Untuk apa dana hasil IPO WBSA akan digunakan? | Seluruh dana hasil Penawaran Umum Perdana Saham setelah dikurangi biaya emisi digunakan untuk mengambilalih saham BIL sebesar Rp215.000.000.000,- dan sisa dana digunakan sebagai modal kerja Perseroan,. Dana modal kerja tersebut tidak digunakan untuk pembelian aset tetap, akuisisi perusahaan lain, atau pembayaran utang,. | Sebesar Rp215.000.000.000,- (dua ratus lima belas miliar Rupiah) akan digunakan Perseroan untuk mengambilalih saham BIL (“Rencana Transaksi”) sebanyak 191.250 (seratus sembilan puluh satu ribu dua ratus lima puluh) lembar saham atau setara dengan 99,99% (sembilan puluh sembilan koma sembilan sembilan persen) dari seluruh modal ditempatkan dan disetor penuh BIL dari BNL (“Saham Yang Dijual”). Sisa dana Penawaran Umum Perdana Saham akan dipergunakan Perseroan sebagai modal kerja Perseroan, meliputi untuk mendukung kegiatan operasional sehari-hari, menjaga likuiditas usaha, serta memperkuat kapasitas layanan logistik multimoda Perseroan.” “Tidak terdapat penggunaan dana pada modal kerja yang digunakan untuk melakukan pembelian aset tetap, melakukan akuisisi perusahaan lain, atau melakukan pembayaran utang Perseroan. |  |

## Simpan Hasil

In [9]:
tables = {
    "L12_1_distribusi.csv": distribusi,
    "L12_2_retrieval_keseluruhan.csv": keseluruhan,
    "L12_3_retrieval_kategori.csv": kategori,
    "retrieval_per_pertanyaan.csv": rincian_retrieval,
    "retrieval_peringkat_chunk.csv": peringkat_retrieval,
    "ragas_ringkasan.csv": ringkasan_ragas,
    "ragas_per_pertanyaan.csv": rincian_ragas,
    "ragas_konteks_per_pertanyaan.csv": konteks_ragas,
}
for filename, rows in tables.items():
    write_csv(HASIL / filename, rows, fieldnames=["No", "ID", "Emiten", "Pertanyaan", "Nomor konteks", "Teks konteks"])

cakupan = (
    f"Dataset berisi {len(rincian_ragas)} pertanyaan dengan jawaban. "
    f"{status_ragas.get('scored', 0)} pertanyaan mendapat empat skor RAGAS. "
    f"{jumlah_tanpa_konteks} pertanyaan tidak membentuk konteks. "
    f"Cakupan penilaian: {cakupan_teks}%."
)
report = "\n\n".join([
    "# Hasil Pengujian Retrieval",
    "## Tabel L12.1 Distribusi Dokumen dan Pertanyaan Pengujian Retrieval",
    markdown_table(distribusi),
    "## Tabel L12.2 Hasil Pengujian Retrieval Keseluruhan",
    markdown_table(keseluruhan),
    "## Tabel L12.3 Hasil Retrieval Berdasarkan Kategori Pertanyaan pada K = 6",
    markdown_table(kategori),
    "## Hasil Pengujian Retrieval per Pertanyaan",
    markdown_table(tabel_retrieval),
    "Peringkat dan teks chunk lengkap tersedia dalam retrieval_peringkat_chunk.csv.",
    "# Hasil Pengujian RAGAS",
    f"Status: {status_evaluasi}.",
    markdown_table(ringkasan_ragas),
    cakupan,
    protokol_teks,
    "## Hasil Pengujian RAGAS per Pertanyaan",
    "### Skor RAGAS per Pertanyaan",
    markdown_table(tabel_skor_ragas),
    "### Jawaban Sistem dan Jawaban Acuan",
    markdown_table(tabel_jawaban_ragas),
    "Konteks yang dikirim ke model tersedia dalam ragas_konteks_per_pertanyaan.csv.",
])
(HASIL / "Hasil_Evaluasi.md").write_text(report + "\n", encoding="utf-8")
for filename in ("Hasil_Evaluasi.md", *tables):
    relative_path = Path(os.path.relpath(HASIL / filename, Path.cwd())).as_posix()
    display(Markdown(f"[{filename}](<{relative_path}>)"))


[Hasil_Evaluasi.md](<evaluation/results/evaluasi_retrieval_ragas_64/Hasil_Evaluasi.md>)

[L12_1_distribusi.csv](<evaluation/results/evaluasi_retrieval_ragas_64/L12_1_distribusi.csv>)

[L12_2_retrieval_keseluruhan.csv](<evaluation/results/evaluasi_retrieval_ragas_64/L12_2_retrieval_keseluruhan.csv>)

[L12_3_retrieval_kategori.csv](<evaluation/results/evaluasi_retrieval_ragas_64/L12_3_retrieval_kategori.csv>)

[retrieval_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas_64/retrieval_per_pertanyaan.csv>)

[retrieval_peringkat_chunk.csv](<evaluation/results/evaluasi_retrieval_ragas_64/retrieval_peringkat_chunk.csv>)

[ragas_ringkasan.csv](<evaluation/results/evaluasi_retrieval_ragas_64/ragas_ringkasan.csv>)

[ragas_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas_64/ragas_per_pertanyaan.csv>)

[ragas_konteks_per_pertanyaan.csv](<evaluation/results/evaluasi_retrieval_ragas_64/ragas_konteks_per_pertanyaan.csv>)